In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [3]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

In [4]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="I'd like to go to London", source="user")
message

TextMessage(id='6fc6a693-d18c-47a5-a0fb-d857a0de52d9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 21, 13, 6, 8, 592701, tzinfo=datetime.timezone.utc), content="I'd like to go to London", type='TextMessage')

In [6]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers.",
    model_client_stream=True
)

In [9]:
from autogen_core import CancellationToken

response = await agent.on_messages([message],cancellation_token=CancellationToken())
response.chat_message.content

'Fantastic! Just remember to pack your umbrella; London has a PhD in “surprise showers"! '

In [12]:
import os
import sqlite3

if os.path.exists("tickets.db"):
    os.remove("tickets.db")


conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()


In [17]:
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [20]:
# Method to get price for a city
def get_city_price(city_name: str) -> float | None:
    """ Get the roundtrip ticket price to travel to the city """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None


In [21]:
get_city_price("Rome")

499.0

In [22]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
        system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
        model_client_stream=True,
        tools=[get_city_price],
        reflect_on_tool_use=True
)

In [23]:
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())


In [24]:
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_ZDP5jdiaEzvjR8z5cWXFjKfN', arguments='{"city_name":"London"}', name='get_city_price')]
[FunctionExecutionResult(content='299.0', name='get_city_price', call_id='call_ZDP5jdiaEzvjR8z5cWXFjKfN', is_error=False)]


'A roundtrip ticket to London will cost you $299! Just make sure to pack your best British accent – you’ll need it for all the tea and crumpets!'